# India Supermarket & Grocery Sales: generate messy + clean data, all inside Colab
No uploads needed. Run the cells top to bottom (Runtime > Run all).

1. Generate a synthetic *true* dataset (5,000 order lines, Apr-Sep 2026, based on the supermarket deck)
2. Corrupt it into a **messy** dataset
3. Clean it back into a **clean** dataset and check it
4. Download both CSVs

In [ ]:
import numpy as np, pandas as pd, random

## 1-2. Generate the true data, then make it messy

In [ ]:
rng = np.random.default_rng(42); random.seed(42)
N = 5000
stores={'ST01':('Mumbai','Metro'),'ST02':('Delhi','Metro'),'ST03':('Bangalore','Metro'),'ST04':('Chennai','Metro'),
        'ST05':('Pune','Tier 2'),'ST06':('Jaipur','Tier 2'),'ST07':('Coimbatore','Tier 2'),'ST08':('Lucknow','Tier 2'),
        'ST09':('Trichy','Tier 3'),'ST10':('Madurai','Tier 3')}
prods={'Staples':[('Basmati Rice 5kg',650),('Atta 10kg',480),('Toor Dal 1kg',165),('Sunflower Oil 1L',150),('Sugar 1kg',52)],
 'Fresh Produce':[('Tomato 1kg',40),('Onion 1kg',45),('Banana 1 dozen',70),('Apple 1kg',210),('Spinach Bunch',30)],
 'Dairy':[('Milk 1L',68),('Curd 400g',45),('Paneer 200g',95),('Butter 500g',285),('Cheese Slices',140)],
 'Snacks':[('Potato Chips',20),('Biscuits Pack',35),('Namkeen 400g',120),('Instant Noodles',15),('Chocolate Bar',50)],
 'Beverages':[('Tea 500g',260),('Coffee 200g',420),('Soft Drink 2L',95),('Fruit Juice 1L',110),('Packaged Water 1L',20)],
 'Household':[('Detergent 1kg',230),('Dishwash Liquid',120),('Floor Cleaner',180),('Toilet Cleaner',95),('Tissue Pack',75)]}
chan_w={'Supermarket':.55,'Online App':.25,'Quick Commerce':.20}
disc={'Supermarket':(17.8,4),'Online App':(16.7,4),'Quick Commerce':(7.5,1.5)}
pay=['UPI','Card','Cash','Wallet','Net Banking']
rows=[]
for i in range(N):
    sid=random.choice(list(stores)); city,tier=stores[sid]
    ch=rng.choice(list(chan_w),p=list(chan_w.values()))
    if tier=='Tier 3' and ch=='Quick Commerce': ch='Supermarket'
    cat=random.choice(list(prods)); p,mrp=random.choice(prods[cat])
    mrp=round(mrp*rng.uniform(.95,1.08)); 
    d=round(float(np.clip(rng.normal(*disc[ch]),0,35)))
    qty=int(rng.choice([1,2,3,4,5,6],p=[.35,.25,.15,.1,.08,.07]))
    net=round(qty*mrp*(1-d/100),2)
    date=pd.Timestamp('2026-04-01')+pd.Timedelta(days=int(rng.integers(0,183)))
    cust=f"C{rng.integers(1000,2500)}" if rng.random()>.18 else np.nan
    rows.append([f"ORD{100001+i}",date,sid,city,tier,ch,cat,p,'Private Label' if rng.random()<.28 else 'National Brand',
                 qty,mrp,d,net,random.choice(pay) if ch=='Supermarket' else ('UPI' if ch!='Supermarket' and rng.random()<.6 else random.choice(pay[:2]+pay[3:])),cust])
cols=['order_id','order_date','store_id','city','city_tier','channel','category','product','brand_type','quantity','mrp','discount_pct','net_amount','payment_method','customer_id']
truth=pd.DataFrame(rows,columns=cols)

# ---------- make it messy ----------
m=truth.copy().astype(object)
def pick(p): return rng.random()<p
fmts=['%Y-%m-%d','%d/%m/%Y','%d-%b-%Y','%B %d, %Y']
m['order_date']=[d.strftime(random.choices(fmts,[.4,.3,.2,.1])[0]) for d in truth.order_date]
def messy_text(v,variants,p=.25):
    if pick(p):
        v=random.choice(variants.get(v,[v])+[v.upper(),v.lower(),' '+v,v+'  ',v.title()])
    return v
cityv={'Bangalore':['Bengaluru','bangalore ','BLR'],'Delhi':['New Delhi','delhi'],'Mumbai':['Bombay','mumbai '],'Coimbatore':['Kovai'],'Trichy':['Tiruchirappalli','Tiruchirapalli']}
chv={'Supermarket':['Super Market','supermarket','Store'],'Online App':['online-app','Online app','App'],'Quick Commerce':['Q-Commerce','quick commerce','QuickCommerce']}
catv={'Fresh Produce':['Fresh Produce','Fresh produce','Fresh Prod.'],'Household':['House hold','Household Items'],'Beverages':['Beverage','Bevarages'],'Staples':['Staple','Stapels']}
m['city']=[messy_text(v,cityv) for v in truth.city]
m['channel']=[messy_text(v,chv) for v in truth.channel]
m['category']=[messy_text(v,catv,.2) for v in truth.category]
m['brand_type']=[random.choice({'Private Label':['PL','Own Brand','private label'],'National Brand':['National','NB','national brand']}[v]+[v]*3) if pick(.25) else v for v in truth.brand_type]
m['payment_method']=[random.choice({'UPI':['upi','Upi ','UPI/QR'],'Card':['Credit Card','Debit Card','card'],'Cash':['cash','CASH'],'Wallet':['wallet','E-Wallet'],'Net Banking':['NetBanking','net banking']}[v]) if pick(.25) else v for v in truth.payment_method]
m['mrp']=[random.choice([f"Rs {v}",f"₹{v:,.2f}",f"Rs.{v}",str(v)]) if pick(.35) else v for v in truth.mrp]
m['discount_pct']=[random.choice([f"{v}%",f"{v} %",f"{v}.0"]) if pick(.35) else v for v in truth.discount_pct]
m['net_amount']=[f"{v:,.2f}" if pick(.3) else v for v in truth.net_amount]
m['quantity']=[random.choice([f"{v} pcs",f"{v}.0",f"{v} units"]) if pick(.12) else v for v in truth.quantity]
m['customer_id']=[random.choice(['N/A','','-','NULL','na','Guest']) if pd.isna(v) else v for v in truth.customer_id]
m['store_id']=[random.choice([v.lower(),v.replace('ST','ST-'),' '+v]) if pick(.12) else v for v in truth.store_id]
# missing values (all recoverable)
for col,p in [('city',.05),('city_tier',.08),('net_amount',.04),('discount_pct',.05)]:
    idx=[i for i in range(N) if pick(p)]
    for i in idx:
        if col=='net_amount' and (m.at[i,'discount_pct'] is None): continue
        m.at[i,col]=random.choice([np.nan,'','N/A','-'])
bad=(m.net_amount.isin([np.nan,'','N/A','-']))&(m.discount_pct.isin([np.nan,'','N/A','-']))
m.loc[bad,'discount_pct']=truth.loc[bad,'discount_pct']
tier_v={'Metro':['metro','METRO','Tier 1'],'Tier 2':['Tier-2','tier 2','T2'],'Tier 3':['Tier-3','tier 3','T3']}
m['city_tier']=[random.choice(tier_v[t]) if (not pd.isna(c) and c not in('','N/A','-') and pick(.2)) else c for c,t in zip(m.city_tier,truth.city_tier)]
# injected invalid rows: bad quantity / zero mrp / future date / test orders
inv=[]
for k in range(60):
    r=m.sample(1,random_state=int(rng.integers(1e6))).iloc[0].copy()
    r['order_id']=f"ORD{200001+k}"
    kind=k%4
    if kind==0: r['quantity']=random.choice([-1,-3,0])
    elif kind==1: r['quantity']=random.choice([999,500])
    elif kind==2: r['mrp']=0
    else: r['order_date']='2027-01-15'
    inv.append(r)
m=pd.concat([m,pd.DataFrame(inv)],ignore_index=True)
# exact duplicates + stray blank row
m=pd.concat([m,m.sample(150,random_state=7)],ignore_index=True)
m=m.sample(frac=1,random_state=3).reset_index(drop=True)

messy = m.copy()
truth_df = truth.copy()
print('messy shape:', messy.shape)
messy.head(10)

## 3. Cleaning function

In [ ]:
NULLS = ['', ' ', 'N/A', 'n/a', 'NA', 'na', 'NULL', 'null', '-', 'Guest']

def clean(df):
    df = df.copy()
    df = df.replace(NULLS, np.nan)
    df = df.dropna(how='all').drop_duplicates()
    df = df.apply(lambda s: s.str.strip() if pd.api.types.is_string_dtype(s) else s)

    # ids
    df['store_id'] = (df['store_id'].str.upper().str.replace('-', '', regex=False))

    # dates (mixed formats)
    DATE_FORMATS = ('%Y-%m-%d', '%d/%m/%Y', '%d-%b-%Y', '%B %d, %Y')
    def parse_date(v):
        for f in DATE_FORMATS:
            d = pd.to_datetime(v, format=f, errors='coerce')
            if pd.notna(d):
                return d
        return pd.NaT
    df['order_date'] = df['order_date'].map(parse_date)

    # categorical standardisation
    city_map = {'bengaluru':'Bangalore','blr':'Bangalore','bangalore':'Bangalore','new delhi':'Delhi','delhi':'Delhi',
                'bombay':'Mumbai','mumbai':'Mumbai','kovai':'Coimbatore','coimbatore':'Coimbatore',
                'tiruchirappalli':'Trichy','tiruchirapalli':'Trichy','trichy':'Trichy'}
    df['city'] = df['city'].map(lambda v: city_map.get(v.lower(), v.title()) if pd.notna(v) else v)

    chan_map = {'supermarket':'Supermarket','super market':'Supermarket','store':'Supermarket',
                'online app':'Online App','online-app':'Online App','app':'Online App',
                'quick commerce':'Quick Commerce','q-commerce':'Quick Commerce','quickcommerce':'Quick Commerce'}
    df['channel'] = df['channel'].map(lambda v: chan_map.get(v.lower(), v) if pd.notna(v) else v)

    cat_map = {'fresh produce':'Fresh Produce','fresh prod.':'Fresh Produce','household':'Household','house hold':'Household',
               'household items':'Household','beverages':'Beverages','beverage':'Beverages','bevarages':'Beverages',
               'staples':'Staples','staple':'Staples','stapels':'Staples','dairy':'Dairy','snacks':'Snacks'}
    df['category'] = df['category'].map(lambda v: cat_map.get(v.lower(), v) if pd.notna(v) else v)

    brand_map = {'private label':'Private Label','pl':'Private Label','own brand':'Private Label',
                 'national brand':'National Brand','national':'National Brand','nb':'National Brand'}
    df['brand_type'] = df['brand_type'].map(lambda v: brand_map.get(v.lower(), v) if pd.notna(v) else v)

    pay_map = {'upi':'UPI','upi/qr':'UPI','card':'Card','credit card':'Card','debit card':'Card','cash':'Cash',
               'wallet':'Wallet','e-wallet':'Wallet','net banking':'Net Banking','netbanking':'Net Banking'}
    df['payment_method'] = df['payment_method'].map(lambda v: pay_map.get(v.lower(), v) if pd.notna(v) else v)

    # numbers stored as text
    num = lambda s: pd.to_numeric(
        s.astype(str).str.replace(r'(?i)rs\.?|₹|pcs|units', '', regex=True)
                     .str.replace(r'[^0-9.\-]', '', regex=True), errors='coerce')
    for c in ['quantity','mrp','discount_pct','net_amount']:
        df[c] = num(df[c])

    # drop invalid rows
    df = df[(df.quantity > 0) & (df.quantity <= 50) & (df.mrp > 0) & (df.order_date <= '2026-09-30')]

    # fill gaps using other columns
    tier_map = {'metro':'Metro','tier 1':'Metro','tier 2':'Tier 2','tier-2':'Tier 2','t2':'Tier 2',
                'tier 3':'Tier 3','tier-3':'Tier 3','t3':'Tier 3'}
    df['city_tier'] = df['city_tier'].map(lambda v: tier_map.get(v.lower(), v) if pd.notna(v) else v)
    store_city = df.dropna(subset=['city']).drop_duplicates('store_id').set_index('store_id')['city']
    store_tier = df.dropna(subset=['city_tier']).drop_duplicates('store_id').set_index('store_id')['city_tier']
    df['city'] = df['city'].fillna(df['store_id'].map(store_city))
    df['city_tier'] = df['city_tier'].fillna(df['store_id'].map(store_tier))
    df['discount_pct'] = df['discount_pct'].fillna(((1 - df['net_amount']/(df['quantity']*df['mrp']))*100).round())
    df['net_amount'] = df['net_amount'].fillna((df['quantity']*df['mrp']*(1-df['discount_pct']/100)).round(2))

    df['quantity'] = df['quantity'].astype(int); df['mrp'] = df['mrp'].astype(int)
    df['discount_pct'] = df['discount_pct'].astype(int); df['net_amount'] = df['net_amount'].round(2)
    return df.sort_values('order_id').reset_index(drop=True)

## 4. Clean the messy data and verify

In [ ]:
# Treat the messy data like a freshly loaded CSV (everything as text)
messy_csv = messy.astype(str).replace({'nan': ''})
clean_df = clean(messy_csv)
print(messy.shape, '->', clean_df.shape)

truth_df['order_date'] = pd.to_datetime(truth_df['order_date'])
truth_df = truth_df.sort_values('order_id').reset_index(drop=True)
same = clean_df.fillna('x').astype(str).values == truth_df.fillna('x').astype(str).values
print('Cleaned data matches the original true data:', same.all())
print('Missing values:', clean_df.isna().sum()[lambda s: s > 0].to_dict(), '(customer_id blank = walk-in)')
clean_df.head(10)

## 5. Quick analysis

In [ ]:
print(clean_df.groupby('channel').discount_pct.mean().round(1).rename('avg discount %'), '\n')
print('Private label share of sales: %.1f%%' % (100*clean_df.loc[clean_df.brand_type=='Private Label','net_amount'].sum()/clean_df.net_amount.sum()))
import matplotlib.pyplot as plt
clean_df.pivot_table(index='city_tier', columns='channel', values='net_amount', aggfunc='sum').plot(kind='bar', figsize=(8,4))
plt.ylabel('Net sales (Rs)'); plt.title('Sales by city tier and channel'); plt.xticks(rotation=0); plt.show()

## 6. Visualisation dashboard
Six charts: monthly sales, discount by channel, city tier, category, private label share, payment mix.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ---- style: recessive grid, thin marks, text in ink colours (not series colours) ----
INK, INK2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
CH_COLOR = {'Supermarket': '#2a78d6', 'Online App': '#eb6834', 'Quick Commerce': '#1baf7a'}  # fixed per channel
CH_ORDER = list(CH_COLOR)
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'axes.edgecolor': GRID,
    'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2, 'text.color': INK,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID,
    'grid.linewidth': .8, 'axes.axisbelow': True, 'font.size': 10, 'axes.titlesize': 12,
    'axes.titleweight': 'bold', 'axes.titlelocation': 'left'})
lakh = mtick.FuncFormatter(lambda v, _: f'{v/1e5:.1f}L')   # Rs lakh

d = clean_df.copy()
d['month'] = d.order_date.dt.to_period('M').dt.to_timestamp()

fig, ax = plt.subplots(2, 3, figsize=(17, 9.5))
fig.suptitle('India supermarket & grocery sales, Apr-Sep 2026 (5,000 order lines)', x=0.01, ha='left',
             fontsize=15, fontweight='bold')

# 1 Monthly net sales by channel (line)
a = ax[0, 0]
m = d.pivot_table(index='month', columns='channel', values='net_amount', aggfunc='sum')[CH_ORDER]
for ch in CH_ORDER:
    a.plot(m.index, m[ch], color=CH_COLOR[ch], lw=2, marker='o', ms=5, mec=SURFACE, mew=1.5, label=ch)
    a.annotate(ch, (m.index[-1], m[ch].iloc[-1]), xytext=(6, 0), textcoords='offset points', va='center', fontsize=9, color=INK2)
a.set_title('Monthly net sales by channel'); a.yaxis.set_major_formatter(lakh); a.set_ylabel('Rs lakh')
a.set_xticks(m.index); a.set_xticklabels([x.strftime('%b') for x in m.index]); a.set_xlim(m.index[0], m.index[-1] + (m.index[-1]-m.index[0])*.22)
a.legend(frameon=False, loc='upper center', bbox_to_anchor=(.5, -.1), ncol=3, fontsize=8, handlelength=1.2)

# 2 Average discount to MRP by channel (bars)
a = ax[0, 1]
disc = d.groupby('channel').discount_pct.mean().reindex(CH_ORDER)
b = a.bar(disc.index, disc.values, color=[CH_COLOR[c] for c in disc.index], width=.55)
for r, v in zip(b, disc.values): a.text(r.get_x()+r.get_width()/2, v+.4, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')
a.set_title('Average discount to MRP'); a.set_ylim(0, disc.max()*1.2); a.set_yticks(range(0, 21, 5)); a.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); a.grid(axis='x', visible=False)

# 3 Sales by city tier and channel (grouped bars)
a = ax[0, 2]
t = d.pivot_table(index='city_tier', columns='channel', values='net_amount', aggfunc='sum')[CH_ORDER].loc[['Metro', 'Tier 2', 'Tier 3']]
w = .26
for i, ch in enumerate(CH_ORDER):
    a.bar([x + (i-1)*(w+.02) for x in range(3)], t[ch], width=w, color=CH_COLOR[ch], label=ch)
a.set_xticks(range(3)); a.set_xticklabels(t.index); a.yaxis.set_major_formatter(lakh); a.set_ylabel('Rs lakh')
a.set_title('Net sales by city tier and channel'); a.legend(frameon=False, fontsize=8, ncol=3, loc='upper center', bbox_to_anchor=(.5, -.1)); a.grid(axis='x', visible=False)

# 4 Sales by category (sorted horizontal bars)
a = ax[1, 0]
cat = d.groupby('category').net_amount.sum().sort_values()
b = a.barh(cat.index, cat.values, color='#2a78d6', height=.6)
for r, v in zip(b, cat.values): a.text(v + cat.max()*.01, r.get_y()+r.get_height()/2, f'{v/1e5:.1f}L', va='center', fontsize=9, color=INK2)
a.set_title('Net sales by category'); a.xaxis.set_major_formatter(lakh); a.set_xlim(0, cat.max()*1.12); a.grid(axis='y', visible=False)

# 5 Private label share of sales by category
a = ax[1, 1]
pl = (d[d.brand_type == 'Private Label'].groupby('category').net_amount.sum() / d.groupby('category').net_amount.sum() * 100).sort_values()
b = a.barh(pl.index, pl.values, color='#2a78d6', height=.6)
overall = 100*d.loc[d.brand_type == 'Private Label', 'net_amount'].sum()/d.net_amount.sum()
for r, v in zip(b, pl.values): a.text(v+.3, r.get_y()+r.get_height()/2, f'{v:.1f}%', va='center', fontsize=9, color=INK2)
a.set_title(f'Private label share of sales (overall {overall:.1f}%)'); a.set_xlim(0, pl.max()*1.2); a.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); a.grid(axis='y', visible=False)

# 6 Payment method mix by channel (100% stacked bars, 2px surface gaps)
a = ax[1, 2]
pm = d.pivot_table(index='channel', columns='payment_method', values='order_id', aggfunc='count').loc[CH_ORDER[::-1]]
pm = pm.fillna(0)
pm = pm.div(pm.sum(axis=1), axis=0) * 100
pm_colors = {'UPI': '#2a78d6', 'Card': '#eb6834', 'Cash': '#1baf7a', 'Wallet': '#eda100', 'Net Banking': '#e87ba4'}
left = pd.Series(0.0, index=pm.index)
for col in ['UPI', 'Card', 'Cash', 'Wallet', 'Net Banking']:
    a.barh(pm.index, pm[col], left=left, color=pm_colors[col], height=.55, edgecolor=SURFACE, linewidth=2, label=col)
    for y, (l, v) in enumerate(zip(left, pm[col])):
        if v > 5: a.text(l+v/2, y, f'{v:.0f}%', ha='center', va='center', fontsize=8.5, color='white' if col == 'UPI' else INK)
    left += pm[col]
a.set_title('Payment method mix by channel'); a.set_xlim(0, 100); a.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
a.legend(frameon=False, ncol=5, fontsize=8, loc='upper center', bbox_to_anchor=(.5, -.1), handlelength=1.2, columnspacing=1); a.grid(axis='y', visible=False)

fig.tight_layout(rect=(0, 0, 1, .95))
fig.savefig('grocery_sales_charts.png', dpi=150)
plt.show()

## 6. Save and download both files

In [ ]:
messy.to_csv('grocery_sales_messy.csv', index=False)
clean_df.to_csv('grocery_sales_clean.csv', index=False)
try:
    from google.colab import files
    files.download('grocery_sales_messy.csv'); files.download('grocery_sales_clean.csv')
except ImportError:
    print('Saved to the current folder (not running in Colab).')